In [6]:
from bs4 import BeautifulSoup
import json
from tqdm import tqdm

In [2]:
# Load your HTML (the full <li> you posted)
with open("wnba_prices.html", "r", encoding="utf-8") as f:
    html_content = f.read()

In [3]:
html_content[:1000]

'<div class="layoutPresenters__OuterMargin-sc-f4172277-1 lnYDpf"><hr class="layoutPresenters__Divider-sc-f4172277-6 gSLyJh"><div class="layoutPresenters__SectionSpacer-sc-f4172277-7 XclfL"><div class="layoutPresenters__Columns-sc-f4172277-2 ksBaUz"><main class="layoutPresenters__MainColumn-sc-f4172277-4 fmxXyL"><div class="layoutPresenters__HeadingRow-sc-f4172277-5 edMHfI"><h2 class="Category__StyledH2-sc-63d0a570-0 kEPKwi">All WNBA Games</h2></div><div><ul class="EventList__BaseEventList-sc-9ea8f928-2 dASQDu"><li data-testid="event-17411342" class="TwoRowEventTitleEventItem__EventItemWrapper-sc-52143443-12 hxKrwI"><a data-testid="event-link-17411342" target="_self" class="TwoRowEventTitleEventItem__ItemLink-sc-52143443-3 ebXXXM" href="/las-vegas-aces-tickets/wnba/2025-05-02-7-pm/17411342"><span class="TwoRowEventTitleEventItem__HiddenScreenReaderDescription-sc-52143443-0 hROKjX"> Find tickets from 20 dollars to Preseason: Las Vegas Aces vs Dallas Wings on Friday May 2 at 7:00 pm at Pu

In [7]:
soup = BeautifulSoup(html_content, "html.parser")


events = []

# Find all event list items
event_items = soup.find_all("li", {"class": "TwoRowEventTitleEventItem__EventItemWrapper-sc-52143443-12"})

for item in tqdm(event_items):
    date = item.find("p", {"data-testid": "date"})
    time = item.find("p", {"data-testid": "time"})
    title = item.find("p", {"data-testid": "event-item-title"})
    description = item.find("p", {"data-testid": "description"})
    link_tag = item.find("a", {"data-testid": lambda x: x and x.startswith("event-link-")})

    if description:
        # Try to parse the location and price from the description text
        desc_text = description.get_text(separator="·")
        parts = [part.strip() for part in desc_text.split('·')]

        if len(parts) >= 2:
            price = parts[0]
            location = " · ".join(parts[1:])  # Combine if there are multiple '·'s
        else:
            price = None
            location = None
    else:
        price = None
        location = None

    event_info = {
        "date": date.get_text(strip=True) if date else None,
        "time": time.get_text(strip=True) if time else None,
        "title": title.get_text(strip=True) if title else None,
        "price": price,
        "location": location,
        "url": link_tag['href'] if link_tag else None
    }

    events.append(event_info)

100%|██████████| 302/302 [00:00<00:00, 19473.90it/s]

{'date': 'May 2', 'time': 'Fri · 7:00pm', 'title': 'Preseason: Las Vegas Aces vs Dallas Wings', 'price': '$20', 'location': ' ·  · Purcell Pavilion at the Joyce Center ·  ·  · Notre Dame, IN', 'url': '/las-vegas-aces-tickets/wnba/2025-05-02-7-pm/17411342'}
{'date': 'May 2', 'time': 'Fri · 7:00pm', 'title': 'Preseason: Brazil Womens Basketball National Team at Chicago Sky', 'price': '$18', 'location': ' ·  · Pete Maravich Assembly Center ·  ·  · Baton Rouge, LA', 'url': '/chicago-sky-tickets/wnba/2025-05-02-7-pm/17367444'}
{'date': 'May 3', 'time': 'Sat · 1:00pm', 'title': 'Preseason: Washington Mystics at Indiana Fever', 'price': '$10', 'location': ' ·  · Gainbridge Fieldhouse ·  ·  · Indianapolis, IN', 'url': '/indiana-fever-tickets/wnba/2025-05-03-1-pm/17317444'}
{'date': 'May 4', 'time': 'Sun · 3:00pm', 'title': 'Preseason: Brazil Womens Basketball National Team at Indiana Fever', 'price': '$146', 'location': ' ·  · Carver-Hawkeye Arena ·  ·  · Iowa City, IA', 'url': '/indiana-fever

In [9]:
# Print extracted events
for event in events:
    print(event)
    print('---')

{'date': 'May 2', 'time': 'Fri · 7:00pm', 'title': 'Preseason: Las Vegas Aces vs Dallas Wings', 'price': '$20', 'location': ' ·  · Purcell Pavilion at the Joyce Center ·  ·  · Notre Dame, IN', 'url': '/las-vegas-aces-tickets/wnba/2025-05-02-7-pm/17411342'}
---
{'date': 'May 2', 'time': 'Fri · 7:00pm', 'title': 'Preseason: Brazil Womens Basketball National Team at Chicago Sky', 'price': '$18', 'location': ' ·  · Pete Maravich Assembly Center ·  ·  · Baton Rouge, LA', 'url': '/chicago-sky-tickets/wnba/2025-05-02-7-pm/17367444'}
---
{'date': 'May 3', 'time': 'Sat · 1:00pm', 'title': 'Preseason: Washington Mystics at Indiana Fever', 'price': '$10', 'location': ' ·  · Gainbridge Fieldhouse ·  ·  · Indianapolis, IN', 'url': '/indiana-fever-tickets/wnba/2025-05-03-1-pm/17317444'}
---
{'date': 'May 4', 'time': 'Sun · 3:00pm', 'title': 'Preseason: Brazil Womens Basketball National Team at Indiana Fever', 'price': '$146', 'location': ' ·  · Carver-Hawkeye Arena ·  ·  · Iowa City, IA', 'url': '/i

In [15]:
import re
import pandas as pd
from datetime import datetime

# Extract home/away teams
def extract_teams(title: str):
    title_clean = re.sub(r'^[^:]+:\s*', '', title)  # Remove game type prefix if it exists
    if ' at ' in title_clean:
        teams = title_clean.split(' at ', 1)
        away_team, home_team = teams[0].strip(), teams[1].strip()
    elif ' vs ' in title_clean:
        teams = title_clean.split(' vs ', 1)
        home_team, away_team = teams[0].strip(), teams[1].strip()
    else:
        home_team, away_team = None, None
    return home_team, away_team

# Extract game type
def extract_game_type(title: str):
    if ':' in title:
        prefix = title.split(':', 1)[0].strip()
        return prefix.lower().replace(' ', '_')
    else:
        return 'regular_season'

# Extract ticket price
def extract_price(price_str: str):
    match = re.search(r'\$(\d+)', price_str)
    return int(match.group(1)) if match else None

# Parse datetime
def parse_datetime(date_str: str, time_str: str):
    try:
        _, time_part = time_str.split('·')
        time_part = time_part.strip()
        full_str = f"{date_str} {datetime.now().year} {time_part}"
        dt = datetime.strptime(full_str, '%b %d %Y %I:%M%p')
        return dt
    except Exception as e:
        print(f"Error parsing datetime for {date_str} {time_str}: {e}")
        return None

# Build the rows
rows = []

for event in events:
    home_team, away_team = extract_teams(event['title'])
    game_type = extract_game_type(event['title'])
    ticket_price = extract_price(event['price'])
    dt = parse_datetime(event['date'], event['time'])
    
    if dt:
        rows.append({
            'home_team': home_team,
            'away_team': away_team,
            'ticket_price': ticket_price,
            'date': dt.date(),
            'day_of_week': dt.strftime('%A'),
            'time': dt.strftime('%H:%M'),
            'location': event['location'].strip(' ·'),
            'url': event['url'],
            'game_type': game_type,
        })

# Create the DataFrame
df = pd.DataFrame(rows)

In [39]:
# this needs to be done for some manual fixes in xlsx file
# df.to_csv("data/wnba_tickets/wnba_prices.csv", index=False)

In [75]:
df = pd.read_csv("data/wnba_tickets/wnba_prices.csv")
df

,home_team,away_team,ticket_price,date,day_of_week,time,location,url,game_type,duplicate
0,Las Vegas Aces,Dallas Wings,20.0,2025-05-02,Friday,19:00,Purcell Pavilion at the Joyce Center · · · N...,/las-vegas-aces-tickets/wnba/2025-05-02-7-pm/1...,preseason,NaN
1,Chicago Sky,Brazil Womens Basketball National Team,18.0,2025-05-02,Friday,19:00,Pete Maravich Assembly Center · · · Baton Ro...,/chicago-sky-tickets/wnba/2025-05-02-7-pm/1736...,preseason,NaN
2,Indiana Fever,Washington Mystics,10.0,2025-05-03,Saturday,13:00,"Gainbridge Fieldhouse · · · Indianapolis, IN",/indiana-fever-tickets/wnba/2025-05-03-1-pm/17...,preseason,NaN
3,Indiana Fever,Brazil Womens Basketball National Team,146.0,2025-05-04,Sunday,15:00,"Carver-Hawkeye Arena · · · Iowa City, IA",/indiana-fever-tickets/wnba/2025-05-04-3-pm/17...,preseason,NaN
4,Seattle Storm,Connecticut Sun,8.0,2025-05-04,Sunday,15:00,"Climate Pledge Arena · · · Seattle, WA",/seattle-storm-tickets/wnba/2025-05-04-3-pm/17...,preseason,NaN
...,...,...,...,...,...,...,...,...,...,...
296,Connecticut Sun,Atlanta Dream,15.0,2025-09-10,Wednesday,19:00,"Mohegan Sun Arena · · · Uncasville, CT",/connecticut-sun-tickets/wnba/2025-09-10-7-pm/...,regular_season,NaN
297,Chicago Sky,New York Liberty,47.0,2025-09-11,Thursday,19:00,"Wintrust Arena · · · Chicago, IL",/chicago-sky-tickets/wnba/2025-09-11-7-pm/1729...,regular_season,NaN
298,Dallas Wings,Phoenix Mercury,63.0,2025-09-11,Thursday,19:00,"College Park Center · · · Arlington, TX",/dallas-wings-tickets/wnba/2025-09-11-7-pm/172...,regular_season,NaN
299,Minnesota Lynx,Golden State Valkyries,132.0,2025-09-11,Thursday,19:00,"Target Center · · · Minneapolis, MN",/minnesota-lynx-tickets/wnba/2025-09-11-7-pm/1...,regular_season,NaN


In [77]:
# Filter only regular season games
regular_season_df = df[(df['game_type'] == 'regular_season') | (df['game_type'] == 'commissioners_cup')]

# Stack home and away teams into one long column
teams = pd.concat([
    regular_season_df[['home_team', 'ticket_price']].rename(columns={'home_team': 'team'}),
    regular_season_df[['away_team', 'ticket_price']].rename(columns={'away_team': 'team'})
])

# Group by team and calculate the average ticket price
average_price_per_team_regular_season = teams.groupby('team')['ticket_price'].mean().reset_index()

# 2. Average ticket price for each home team (for all games)
average_price_per_home_team = regular_season_df.groupby('home_team')['ticket_price'].mean().reset_index()

# Show the results
print("Average ticket price per team (regular season, home or away):")
print(average_price_per_team_regular_season.sort_values(by='ticket_price', ascending=False))
average_price_per_team_regular_season.sort_values(by='ticket_price', ascending=False).to_csv("data/wnba_tickets/output/average_price.csv")

print("\nAverage ticket price per home team (all games):")
print(average_price_per_home_team.sort_values(by='ticket_price', ascending=False))
average_price_per_home_team.sort_values(by='ticket_price', ascending=False).to_csv("data/wnba_tickets/output/average_home_price.csv")

average_price_per_matchup = regular_season_df.groupby(['home_team', 'away_team'])['ticket_price'].mean().reset_index()

# Show it
print("Average ticket price for each home vs away matchup:")
print(average_price_per_matchup.sort_values(by='ticket_price', ascending=False))
average_price_per_matchup.sort_values(by='ticket_price', ascending=False).to_csv("data/wnba_tickets+output/average_matchup_price.csv")

Average ticket price per team (regular season, home or away):
                      team  ticket_price
3             Dallas Wings     72.636364
8           Minnesota Lynx     68.431818
5            Indiana Fever     68.227273
1              Chicago Sky     52.204545
12      Washington Mystics     51.886364
4   Golden State Valkyries     46.318182
0            Atlanta Dream     43.186047
9         New York Liberty     42.750000
7       Los Angeles Sparks     40.113636
6           Las Vegas Aces     33.568182
2          Connecticut Sun     33.227273
11           Seattle Storm     30.395349
10         Phoenix Mercury     28.954545

Average ticket price per home team (all games):
                 home_team  ticket_price
8           Minnesota Lynx    108.863636
12      Washington Mystics     71.000000
3             Dallas Wings     65.727273
0            Atlanta Dream     60.380952
4   Golden State Valkyries     53.363636
1              Chicago Sky     48.090909
7       Los Angeles Sparks  

OSError: Cannot save file into a non-existent directory: 'data/wnba_tickets+output'

In [78]:
# 1. Overall average ticket price (across all regular season + cup games)
overall_avg_price = regular_season_df['ticket_price'].mean()

# 2. Find all unique away teams
away_teams = regular_season_df['away_team'].unique()

# 3. For each away team, calculate their visitor impact
impact_rows = []

for team in away_teams:
    team_games = regular_season_df[regular_season_df['away_team'] == team]
    team_avg_price = team_games['ticket_price'].mean()
    
    if pd.notna(team_avg_price):
        percent_difference = ((team_avg_price - overall_avg_price) / overall_avg_price) * 100
        impact_rows.append({
            'away_team': team,
            'avg_ticket_price_when_visiting': round(team_avg_price, 2),
            'percent_difference_from_overall': round(percent_difference, 2)
        })

# 4. Create a DataFrame
visitor_impact_df = pd.DataFrame(impact_rows)

# 5. Sort by biggest impact
visitor_impact_df = visitor_impact_df.sort_values(by='percent_difference_from_overall', ascending=False)

# 6. Show the final table
visitor_impact_df

visitor_impact_df.to_csv("data/wnba_tickets/output/visitor_impact.csv") 

In [79]:
# 1. Create a matchup key that is always Team A vs Team B (sorted alphabetically)
def create_matchup(row):
    teams = sorted([row['home_team'], row['away_team']])
    return f"{teams[0]} vs {teams[1]}"

# Add the matchup column
regular_season_df['matchup'] = regular_season_df.apply(create_matchup, axis=1)

# 2. Group by matchup and calculate average ticket price
matchup_prices = regular_season_df.groupby('matchup')['ticket_price'].mean().reset_index()

# 3. Sort by average price descending
matchup_prices = matchup_prices.sort_values(by='ticket_price', ascending=False)

# 4. Show the most expensive matchup
most_expensive_matchup = matchup_prices.iloc[0]

print(f"The most expensive matchup is {most_expensive_matchup['matchup']} with an average ticket price of ${most_expensive_matchup['ticket_price']:.2f}")

matchup_prices.to_csv("data/wnba_tickets/output/matchup_prices.csv", index=False)

matchup_prices

The most expensive matchup is Dallas Wings vs Washington Mystics with an average ticket price of $123.67


/var/folders/0l/bxmy5l011594gflgwzxxndhr0000gn/T/ipykernel_65274/244174067.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  regular_season_df['matchup'] = regular_season_df.apply(create_matchup, axis=1)


,matchup,ticket_price
41,Dallas Wings vs Washington Mystics,123.666667
42,Golden State Valkyries vs Indiana Fever,111.000000
2,Atlanta Dream vs Dallas Wings,104.000000
34,Dallas Wings vs Indiana Fever,95.750000
13,Chicago Sky vs Dallas Wings,88.666667
...,...,...
61,Las Vegas Aces vs Seattle Storm,16.000000
26,Connecticut Sun vs Las Vegas Aces,15.500000
30,Connecticut Sun vs Phoenix Mercury,14.000000
60,Las Vegas Aces vs Phoenix Mercury,13.500000


In [74]:
regular_season_df.sort_values(by='ticket_price', ascending=False).tail(10)

# regular_season_df.sort_values(by='ticket_price', ascending=False).to_csv('data/wnba_tickets/output/wnba_prices_sorted.csv', index=False)

,home_team,away_team,ticket_price,date,day_of_week,time,location,url,game_type,duplicate,matchup
256,New York Liberty,Connecticut Sun,10.0,2025-08-25,Monday,19:00,"Barclays Center · · · Brooklyn, NY",/new-york-liberty-tickets/wnba/2025-08-25-7-pm...,regular_season,NaN,Connecticut Sun vs New York Liberty
109,Las Vegas Aces,Washington Mystics,10.0,2025-06-26,Thursday,19:00,Michelob ULTRA Arena at Mandalay Bay Resort & ...,/las-vegas-aces-tickets/wnba/2025-06-26-7-pm/1...,regular_season,NaN,Las Vegas Aces vs Washington Mystics
106,Las Vegas Aces,Connecticut Sun,9.0,2025-06-25,Wednesday,19:00,Michelob ULTRA Arena at Mandalay Bay Resort & ...,/las-vegas-aces-tickets/wnba/2025-06-25-7-pm/1...,regular_season,NaN,Connecticut Sun vs Las Vegas Aces
262,New York Liberty,Washington Mystics,9.0,2025-08-28,Thursday,19:00,"Barclays Center · · · Brooklyn, NY",/new-york-liberty-tickets/wnba/2025-08-28-7-pm...,regular_season,NaN,New York Liberty vs Washington Mystics
286,Connecticut Sun,Phoenix Mercury,8.0,2025-09-06,Saturday,13:00,"Mohegan Sun Arena · · · Uncasville, CT",/connecticut-sun-tickets/wnba/2025-09-06-1-pm/...,regular_season,NaN,Connecticut Sun vs Phoenix Mercury
48,Phoenix Mercury,Minnesota Lynx,8.0,2025-05-30,Friday,19:00,"PHX Arena · · · Phoenix, AZ",/phoenix-mercury-tickets/wnba/2025-05-30-7-pm/...,regular_season,NaN,Minnesota Lynx vs Phoenix Mercury
161,Las Vegas Aces,Atlanta Dream,8.0,2025-07-22,Tuesday,19:00,Michelob ULTRA Arena at Mandalay Bay Resort & ...,/las-vegas-aces-tickets/wnba/2025-07-22-7-pm/1...,regular_season,NaN,Atlanta Dream vs Las Vegas Aces
244,Las Vegas Aces,Phoenix Mercury,6.0,2025-08-21,Thursday,19:00,Michelob ULTRA Arena at Mandalay Bay Resort & ...,/las-vegas-aces-tickets/wnba/2025-08-21-7-pm/1...,regular_season,NaN,Las Vegas Aces vs Phoenix Mercury
240,Las Vegas Aces,Atlanta Dream,6.0,2025-08-19,Tuesday,19:00,Michelob ULTRA Arena at Mandalay Bay Resort & ...,/las-vegas-aces-tickets/wnba/2025-08-19-7-pm/1...,regular_season,NaN,Atlanta Dream vs Las Vegas Aces
229,Seattle Storm,Atlanta Dream,NaN,2025-08-15,Friday,19:00,"Vancouver, Canada",/wnba-canada-game-tickets/wnba/2025-08-15-7-pm...,regular_season,NaN,Atlanta Dream vs Seattle Storm
